In [3]:
import requests
import pandas as pd

# URL correcta
url = "https://api.open-meteo.com/v1/forecast?latitude=-34.6&longitude=-58.4&hourly=temperature_2m,precipitation&current_weather=true"

# API call
response = requests.get(url)
data = response.json()

# --- CLIMA ACTUAL ---
current = data["current_weather"]
df_current = pd.DataFrame([current])

# --- FORECAST ---
hourly = data["hourly"]

df_forecast = pd.DataFrame({
    "time": hourly["time"],
    "temperature": hourly["temperature_2m"],
    "precipitation": hourly["precipitation"]
})

# --- TIMESTAMPS ---
df_current["time"] = pd.to_datetime(df_current["time"])
df_forecast["time"] = pd.to_datetime(df_forecast["time"])

# --- TIPOS NUMÉRICOS ---
df_forecast["temperature"] = pd.to_numeric(df_forecast["temperature"], errors="coerce")
df_forecast["precipitation"] = pd.to_numeric(df_forecast["precipitation"], errors="coerce")

# --- LIMPIEZA ---
df_forecast = df_forecast.dropna().drop_duplicates()

# ver resultados
df_forecast.head()

,time,temperature,precipitation
0,2026-06-04 00:00:00,15.2,0.0
1,2026-06-04 01:00:00,15.5,0.0
2,2026-06-04 02:00:00,15.2,0.0
3,2026-06-04 03:00:00,14.8,0.0
4,2026-06-04 04:00:00,14.5,0.0


In [4]:
if response.status_code != 200:
    raise Exception("Error en API")

assert not df_forecast.empty, "Forecast vacío"

from datetime import datetime

df_forecast["ingestion_time"] = datetime.utcnow()

df_forecast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   time            168 non-null    datetime64[ns]
 1   temperature     168 non-null    float64       
 2   precipitation   168 non-null    float64       
 3   ingestion_time  168 non-null    datetime64[us]
dtypes: datetime64[ns](1), datetime64[us](1), float64(2)
memory usage: 5.4 KB
